# High-Performance LLM Systems

**Module 15: LLM Systems Design**  
**Objective**: Master production LLM serving and optimization

LLM Systems:
- vLLM and Continuous Batching
- PagedAttention for Memory Management
- Inference Optimization Techniques
- KV Cache Management
- Throughput vs Latency Tradeoffs
- Model Parallelism

## What You'll Learn
1. Why LLM inference is challenging
2. vL LM architecture and continuous batching
3. PagedAttention for efficient memory use
4. KV cache optimization strategies
5. Batching strategies for throughput
6. Production serving best practices

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Dict
from dataclasses import dataclass
import time
from collections import deque

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
np.random.seed(42)

## 1. LLM Inference Challenges

**Key Challenges**:

1. **Memory-bound**: Inference bottleneck is memory bandwidth, not compute
2. **Large KV cache**: Each token stores keys and values (2 × hidden_dim × n_layers)
3. **Variable length**: Sequences have different lengths → memory fragmentation
4. **Sequential generation**: Can't parallelize token generation (autoregressive)
5. **Batch inefficiency**: Static batching wastes resources

**Example**: GPT-3 (175B parameters)
- Model size: ~350GB (FP16)
- KV cache per token: ~800KB
- 100 concurrent users, 1000 tokens each → 80GB just for KV cache!

**Goals**:
- ↑ Throughput: Serve more requests/second
- ↓ Latency: Faster time to first token (TTFT) and inter-token latency
- ↓ Memory: Fit more sequences in GPU memory
- $ Cost: Maximize tokens/dollar

In [ ]:
def visualize_llm_inference_challenges():
    """Visualize LLM inference challenges."""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Memory Bottleneck
    ax1 = axes[0, 0]
    ax1.set_title('Memory vs Compute Bottleneck', fontsize=13, weight='bold')
    
    components = ['Model\nWeights', 'KV\nCache', 'Activations', 'Compute\n(FLOPs)']
    sizes = [350, 80, 20, 5]  # Relative sizes
    colors = ['lightblue', 'lightcoral', 'lightgreen', 'lightyellow']
    
    bars = ax1.bar(components, sizes, color=colors, edgecolor='black', linewidth=2)
    ax1.set_ylabel('Relative Size/Bottleneck', fontsize=11)
    ax1.set_ylim(0, max(sizes) * 1.2)
    
    for i, (bar, size) in enumerate(zip(bars, sizes)):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{size}',
                ha='center', va='bottom', fontsize=11, weight='bold')
    
    ax1.axhline(y=100, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax1.text(0.5, 110, 'Memory bandwidth limit', fontsize=10, color='red',
            style='italic', ha='center')
    
    # 2. KV Cache Growth
    ax2 = axes[0, 1]
    ax2.set_title('KV Cache Memory Growth', fontsize=13, weight='bold')
    
    seq_lengths = np.array([128, 512, 1024, 2048, 4096])
    batch_sizes = [1, 8, 32]
    
    kv_cache_per_token_mb = 0.8  # MB per token (example)
    
    for batch_size in batch_sizes:
        memory = seq_lengths * batch_size * kv_cache_per_token_mb / 1024  # GB
        ax2.plot(seq_lengths, memory, marker='o', linewidth=2,
                markersize=8, label=f'Batch={batch_size}')
    
    ax2.set_xlabel('Sequence Length', fontsize=11)
    ax2.set_ylabel('KV Cache (GB)', fontsize=11)
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=80, color='red', linestyle='--', linewidth=2, alpha=0.5,
               label='A100 80GB limit')
    
    # 3. Static vs Continuous Batching
    ax3 = axes[1, 0]
    ax3.set_title('Static vs Continuous Batching', fontsize=13, weight='bold')
    ax3.set_xlim(0, 10)
    ax3.set_ylim(0, 8)
    ax3.set_xlabel('Time Steps', fontsize=11)
    ax3.set_ylabel('Batch Slot', fontsize=11)
    
    # Static batching
    static_sequences = [
        (0, 5, 0),  # start, end, slot
        (0, 7, 1),
        (0, 3, 2),
        (0, 6, 3),
    ]
    
    for start, end, slot in static_sequences:
        color = plt.cm.Set3(slot)
        rect = plt.Rectangle((start, slot), end-start, 0.8,
                            facecolor=color, edgecolor='black', linewidth=2)
        ax3.add_patch(rect)
        ax3.text((start+end)/2, slot+0.4, f'Seq{slot+1}',
                ha='center', va='center', fontsize=9, weight='bold')
    
    # Wasted computation (gray)
    for slot in range(4):
        max_end = max(end for s, end, sl in static_sequences)
        seq_end = [end for s, end, sl in static_sequences if sl == slot][0]
        if seq_end < max_end:
            rect = plt.Rectangle((seq_end, slot), max_end-seq_end, 0.8,
                                facecolor='gray', edgecolor='red',
                                linewidth=2, linestyle='--', alpha=0.5)
            ax3.add_patch(rect)
            ax3.text((seq_end+max_end)/2, slot+0.4, 'WASTED',
                    ha='center', va='center', fontsize=8,
                    color='red', weight='bold')
    
    ax3.set_yticks(range(4))
    ax3.set_yticklabels([f'Slot {i}' for i in range(4)])
    ax3.text(5, 6.5, 'Static Batching:\nWait for all sequences to finish',
            ha='center', fontsize=10, style='italic',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 4. Throughput vs Latency
    ax4 = axes[1, 1]
    ax4.set_title('Throughput vs Latency Tradeoff', fontsize=13, weight='bold')
    
    batch_sizes = np.array([1, 2, 4, 8, 16, 32, 64, 128])
    
    # Throughput increases with batch size
    throughput = batch_sizes * 100 / (1 + 0.02 * batch_sizes)  # tokens/sec
    
    # Latency increases with batch size
    latency = 10 + 2 * np.log(batch_sizes)  # ms per token
    
    ax4_twin = ax4.twinx()
    
    line1 = ax4.plot(batch_sizes, throughput, 'b-o', linewidth=2,
                    markersize=8, label='Throughput')
    line2 = ax4_twin.plot(batch_sizes, latency, 'r-s', linewidth=2,
                         markersize=8, label='Latency')
    
    ax4.set_xlabel('Batch Size', fontsize=11)
    ax4.set_ylabel('Throughput (tokens/sec)', fontsize=11, color='blue')
    ax4_twin.set_ylabel('Latency (ms/token)', fontsize=11, color='red')
    ax4.tick_params(axis='y', labelcolor='blue')
    ax4_twin.tick_params(axis='y', labelcolor='red')
    ax4.set_xscale('log', base=2)
    ax4.grid(True, alpha=0.3)
    
    # Optimal point annotation
    optimal_idx = 3  # batch_size=8
    ax4.axvline(batch_sizes[optimal_idx], color='green', linestyle='--',
               linewidth=2, alpha=0.5)
    ax4.text(batch_sizes[optimal_idx], max(throughput)*0.5,
            'Sweet\nspot', ha='center', fontsize=10, color='green',
            weight='bold', bbox=dict(boxstyle='round', facecolor='lightgreen'))
    
    plt.tight_layout()
    plt.show()
    
    print("LLM Inference Challenges:")
    print("="*70)
    
    print("\n1. MEMORY BOTTLENECK:")
    print("   • Loading model weights from memory is the slowest operation")
    print("   • KV cache grows linearly with sequence length and batch size")
    print("   • Memory fragmentation wastes precious GPU RAM")
    
    print("\n2. STATIC BATCHING PROBLEMS:")
    print("   • Must wait for all sequences in batch to finish")
    print("   • Shorter sequences waste computation (padding)")
    print("   • Can't add new requests mid-batch")
    print("   • Low GPU utilization")
    
    print("\n3. THROUGHPUT VS LATENCY:")
    print("   • Larger batches → higher throughput")
    print("   • Larger batches → higher latency per request")
    print("   • Need to find sweet spot for your use case")
    print("   • Interactive apps need low latency (batch=1-8)")
    print("   • Batch processing can use large batches (batch=32-128)")
    
    print("\n4. SOLUTIONS:")
    print("   ✓ Continuous batching (vLLM, TGI)")
    print("   ✓ PagedAttention for memory efficiency")
    print("   ✓ KV cache quantization")
    print("   ✓ Speculative decoding")
    print("   ✓ Model quantization (INT8, INT4)")

visualize_llm_inference_challenges()

## 2. vLLM and Continuous Batching

**vLLM** is a high-throughput LLM serving engine

**Key Innovation**: Continuous (Dynamic) Batching

**Static Batching** (traditional):
```
Batch 1: [Req1, Req2, Req3, Req4]
- Start all 4 together
- Wait for longest to finish
- Then start Batch 2
```

**Continuous Batching** (vLLM):
```
Time 0: [Req1, Req2, Req3, Req4]
Time 5: Req3 finishes → [Req1, Req2, Req4, Req5]  ← Add Req5!
Time 7: Req1 finishes → [Req2, Req4, Req5, Req6]  ← Add Req6!
```

**Benefits**:
- No wasted computation
- Always keep batch full
- Much higher GPU utilization (2-4x throughput)
- Lower latency for individual requests

In [ ]:
@dataclass
class Request:
    """Inference request."""
    id: int
    prompt: str
    max_tokens: int
    arrival_time: float
    num_generated: int = 0
    finished: bool = False
    
class ContinuousBatchingScheduler:
    """
    Continuous batching scheduler for LLM inference.
    """
    
    def __init__(self, max_batch_size: int = 32):
        self.max_batch_size = max_batch_size
        self.waiting_queue = deque()
        self.running_requests: List[Request] = []
        self.current_time = 0.0
        
        # Metrics
        self.completed_requests = []
        self.total_tokens_generated = 0
    
    def add_request(self, request: Request):
        """Add new request to waiting queue."""
        self.waiting_queue.append(request)
    
    def schedule(self) -> List[Request]:
        """
        Get next batch to process.
        
        Continuous batching:
        1. Remove finished requests
        2. Fill empty slots with waiting requests
        3. Return current batch
        """
        # Remove finished requests
        newly_finished = [req for req in self.running_requests if req.finished]
        for req in newly_finished:
            self.completed_requests.append(req)
            self.running_requests.remove(req)
        
        # Fill empty slots from waiting queue
        while len(self.running_requests) < self.max_batch_size and self.waiting_queue:
            new_req = self.waiting_queue.popleft()
            self.running_requests.append(new_req)
        
        return self.running_requests
    
    def step(self):
        """
        Simulate one inference step (generate one token for each request in batch).
        """
        batch = self.schedule()
        
        if not batch:
            return
        
        # Simulate token generation
        for req in batch:
            if not req.finished:
                req.num_generated += 1
                self.total_tokens_generated += 1
                
                # Check if finished
                if req.num_generated >= req.max_tokens:
                    req.finished = True
        
        self.current_time += 1
    
    def get_metrics(self) -> Dict:
        """Get performance metrics."""
        if not self.completed_requests:
            return {}
        
        latencies = [self.current_time - req.arrival_time
                    for req in self.completed_requests]
        
        return {
            'completed_requests': len(self.completed_requests),
            'total_tokens': self.total_tokens_generated,
            'avg_latency': np.mean(latencies),
            'throughput': self.total_tokens_generated / self.current_time if self.current_time > 0 else 0,
            'batch_size_avg': np.mean([len(self.running_requests)]) if self.running_requests else 0
        }

# Simulate continuous batching
print("\nContinuous Batching Simulation:")
print("="*70)

scheduler = ContinuousBatchingScheduler(max_batch_size=4)

# Add initial requests
initial_requests = [
    Request(id=1, prompt="Hello", max_tokens=5, arrival_time=0),
    Request(id=2, prompt="How are you?", max_tokens=7, arrival_time=0),
    Request(id=3, prompt="What is AI?", max_tokens=3, arrival_time=0),
    Request(id=4, prompt="Explain quantum", max_tokens=6, arrival_time=0),
]

for req in initial_requests:
    scheduler.add_request(req)

# Add requests during execution
delayed_requests = [
    Request(id=5, prompt="Tell me more", max_tokens=4, arrival_time=5),
    Request(id=6, prompt="Interesting", max_tokens=8, arrival_time=7),
]

print("\nExecution Timeline:")
print("-" * 70)

batch_history = []

for step in range(15):
    # Add delayed requests
    for req in delayed_requests:
        if req.arrival_time == step:
            scheduler.add_request(req)
            print(f"\nTime {step}: New request #{req.id} arrives")
    
    # Get current batch
    batch = scheduler.schedule()
    batch_ids = [req.id for req in batch]
    batch_history.append(len(batch))
    
    print(f"Time {step:2d}: Batch {batch_ids} (size={len(batch)})")
    
    # Show request status
    for req in batch:
        status = "✓ DONE" if req.finished else f"{req.num_generated}/{req.max_tokens}"
        print(f"         Req#{req.id}: {status}")
    
    # Execute step
    scheduler.step()
    
    # Check if all done
    if not scheduler.running_requests and not scheduler.waiting_queue:
        break

# Metrics
metrics = scheduler.get_metrics()
print("\n" + "="*70)
print("METRICS:")
print(f"  Completed requests: {metrics['completed_requests']}")
print(f"  Total tokens: {metrics['total_tokens']}")
print(f"  Average latency: {metrics['avg_latency']:.2f} time steps")
print(f"  Throughput: {metrics['throughput']:.2f} tokens/step")

# Visualize batch size over time
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Continuous Batching: Batch Size Over Time', fontsize=13, weight='bold')
ax.plot(range(len(batch_history)), batch_history, 'b-o', linewidth=2, markersize=8)
ax.set_xlabel('Time Step', fontsize=11)
ax.set_ylabel('Batch Size', fontsize=11)
ax.set_ylim(0, 5)
ax.grid(True, alpha=0.3)
ax.axhline(y=4, color='red', linestyle='--', linewidth=2, alpha=0.5,
          label='Max batch size=4')

# Annotate key events
ax.annotate('Req#3 finishes', xy=(3, batch_history[3]), xytext=(3, 3.5),
           arrowprops=dict(arrowstyle='->', color='green', lw=2),
           fontsize=10, color='green', weight='bold')
ax.annotate('Req#5 added', xy=(5, batch_history[5]), xytext=(5, 3.5),
           arrowprops=dict(arrowstyle='->', color='blue', lw=2),
           fontsize=10, color='blue', weight='bold')

ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("\nCONTINUOUS BATCHING BENEFITS:")
print("  ✓ No wasted computation on finished sequences")
print("  ✓ Batch stays full → higher GPU utilization")
print("  ✓ New requests served faster (no waiting for batch to finish)")
print("  ✓ 2-4x throughput improvement over static batching")

## 3. PagedAttention

**Problem**: KV cache memory management

**Traditional approach**:
- Allocate contiguous memory for max_seq_len
- Wasteful: Most sequences don't reach max length
- Fragmentation: Can't use freed memory efficiently

**PagedAttention** (key innovation of vLLM):
- Inspired by OS virtual memory
- Divide KV cache into blocks (pages)
- Non-contiguous storage
- Share blocks across sequences (for same prefix)

**Benefits**:
- Near-zero waste
- No fragmentation
- Memory sharing (beam search, parallel sampling)
- 2-4x more sequences fit in memory

In [ ]:
class PagedKVCache:
    """
    Paged KV cache manager (simplified vLLM implementation).
    """
    
    def __init__(self, num_blocks: int, block_size: int,
                 num_heads: int, head_dim: int):
        """
        Args:
            num_blocks: Total number of blocks available
            block_size: Tokens per block
            num_heads: Number of attention heads
            head_dim: Dimension per head
        """
        self.num_blocks = num_blocks
        self.block_size = block_size
        self.num_heads = num_heads
        self.head_dim = head_dim
        
        # Physical blocks (actual memory)
        # Shape: (num_blocks, 2, block_size, num_heads, head_dim)
        # 2 for key and value
        self.blocks = torch.zeros(
            num_blocks, 2, block_size, num_heads, head_dim,
            device=device
        )
        
        # Free block IDs
        self.free_blocks = set(range(num_blocks))
        
        # Sequence to block mapping
        self.seq_to_blocks: Dict[int, List[int]] = {}
    
    def allocate_sequence(self, seq_id: int, num_tokens: int) -> bool:
        """
        Allocate blocks for a sequence.
        
        Returns:
            success: Whether allocation succeeded
        """
        num_blocks_needed = (num_tokens + self.block_size - 1) // self.block_size
        
        if len(self.free_blocks) < num_blocks_needed:
            return False  # Out of memory
        
        # Allocate blocks
        allocated_blocks = []
        for _ in range(num_blocks_needed):
            block_id = self.free_blocks.pop()
            allocated_blocks.append(block_id)
        
        self.seq_to_blocks[seq_id] = allocated_blocks
        return True
    
    def free_sequence(self, seq_id: int):
        """Free blocks used by sequence."""
        if seq_id not in self.seq_to_blocks:
            return
        
        blocks = self.seq_to_blocks[seq_id]
        self.free_blocks.update(blocks)
        del self.seq_to_blocks[seq_id]
    
    def get_blocks(self, seq_id: int) -> Optional[List[int]]:
        """Get block IDs for sequence."""
        return self.seq_to_blocks.get(seq_id)
    
    def append_token(self, seq_id: int):
        """
        Append token to sequence (may need to allocate new block).
        """
        if seq_id not in self.seq_to_blocks:
            return False
        
        blocks = self.seq_to_blocks[seq_id]
        tokens_allocated = len(blocks) * self.block_size
        
        # Check if need new block
        # In real implementation, track exact token count
        # Here simplified
        
        return True
    
    def get_utilization(self) -> float:
        """Get memory utilization."""
        used_blocks = self.num_blocks - len(self.free_blocks)
        return used_blocks / self.num_blocks

# Example usage
print("\nPagedAttention KV Cache:")
print("="*70)

# Create paged KV cache
cache = PagedKVCache(
    num_blocks=100,
    block_size=16,  # 16 tokens per block
    num_heads=32,
    head_dim=128
)

print(f"\nCache Configuration:")
print(f"  Total blocks: {cache.num_blocks}")
print(f"  Block size: {cache.block_size} tokens")
print(f"  Max total tokens: {cache.num_blocks * cache.block_size}")
print(f"  Memory per block: {cache.block_size * cache.num_heads * cache.head_dim * 2 * 2 / 1024:.2f} KB")
print(f"  Total memory: {cache.num_blocks * cache.block_size * cache.num_heads * cache.head_dim * 2 * 2 / 1024 / 1024:.2f} MB")

# Allocate sequences
sequences = [
    (1, 50),  # seq_id, num_tokens
    (2, 30),
    (3, 100),
    (4, 20),
]

print("\nAllocating sequences:")
for seq_id, num_tokens in sequences:
    success = cache.allocate_sequence(seq_id, num_tokens)
    blocks = cache.get_blocks(seq_id)
    print(f"  Seq {seq_id} ({num_tokens} tokens): {len(blocks) if blocks else 0} blocks allocated, "
          f"Status: {'✓' if success else '✗'}")

print(f"\nMemory utilization: {cache.get_utilization():.1%}")
print(f"Free blocks: {len(cache.free_blocks)}")

# Free some sequences
print("\nFreeing sequences 2 and 4:")
cache.free_sequence(2)
cache.free_sequence(4)
print(f"Memory utilization: {cache.get_utilization():.1%}")
print(f"Free blocks: {len(cache.free_blocks)}")

# Visualize paged memory
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Traditional contiguous allocation
ax1 = axes[0]
ax1.set_title('Traditional Contiguous Allocation', fontsize=13, weight='bold')
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 4)
ax1.axis('off')

trad_seqs = [
    (0, 4, 3, 'Seq 1 (50 tokens)', 'lightblue'),  # x, y, width, label, color
    (0, 2, 2, 'Seq 2 (30 tokens)', 'lightgreen'),
    (0, 1, 6.5, 'Seq 3 (100 tokens)', 'lightyellow'),
    (0, 0, 1.5, 'Seq 4 (20 tokens)', 'lightcoral'),
]

for x, y, width, label, color in trad_seqs:
    rect = plt.Rectangle((x, y*0.8), width, 0.6,
                         facecolor=color, edgecolor='black', linewidth=2)
    ax1.add_patch(rect)
    ax1.text(x + width/2, y*0.8 + 0.3, label, ha='center', va='center',
            fontsize=9, weight='bold')
    
    # Wasted space
    max_width = 10
    if x + width < max_width:
        waste_rect = plt.Rectangle((x + width, y*0.8), max_width - width - x, 0.6,
                                   facecolor='gray', edgecolor='red',
                                   linewidth=2, linestyle='--', alpha=0.3)
        ax1.add_patch(waste_rect)

ax1.text(5, 3.5, 'Pre-allocated max_seq_len\n→ Wasted memory',
        ha='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Paged allocation
ax2 = axes[1]
ax2.set_title('PagedAttention: Block-based Allocation', fontsize=13, weight='bold')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 4)
ax2.axis('off')

# Draw blocks (each block = 16 tokens)
block_width = 0.8
paged_seqs = [
    ([(0,3), (1,3), (2,3), (3,3)], 'Seq 1 (4 blocks)', 'lightblue'),  # blocks, label, color
    ([(4,3), (5,3)], 'Seq 2 (2 blocks)', 'lightgreen'),
    ([(0,2), (1,2), (2,2), (3,2), (4,2), (5,2), (6,2)], 'Seq 3 (7 blocks)', 'lightyellow'),
    ([(6,3), (7,3)], 'Seq 4 (2 blocks)', 'lightcoral'),
]

for blocks, label, color in paged_seqs:
    for bx, by in blocks:
        rect = plt.Rectangle((bx*block_width, by*0.8), block_width*0.9, 0.6,
                            facecolor=color, edgecolor='black', linewidth=2)
        ax2.add_patch(rect)
    
    # Label
    center_x = np.mean([bx for bx, _ in blocks]) * block_width + block_width/2
    center_y = blocks[0][1] * 0.8 + 0.3
    ax2.text(center_x, center_y, label, ha='center', va='center',
            fontsize=8, weight='bold')

# Free blocks
free_blocks = [(7,2), (8,2), (8,3), (0,1), (1,1), (2,1)]
for bx, by in free_blocks:
    rect = plt.Rectangle((bx*block_width, by*0.8), block_width*0.9, 0.6,
                         facecolor='white', edgecolor='green',
                         linewidth=2, linestyle='--')
    ax2.add_patch(rect)
    ax2.text(bx*block_width + block_width*0.45, by*0.8 + 0.3, 'FREE',
            ha='center', va='center', fontsize=7, color='green', weight='bold')

ax2.text(5, 3.5, 'Block-based allocation\n→ No waste, flexible',
        ha='center', fontsize=10, style='italic',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

plt.tight_layout()
plt.show()

print("\nPAGEDATTENTION BENEFITS:")
print("  ✓ Near-zero memory waste (only last block may have empty slots)")
print("  ✓ No fragmentation (blocks can be non-contiguous)")
print("  ✓ Memory sharing for beam search / parallel sampling")
print("  ✓ 2-4x more sequences fit in memory")
print("  ✓ Dynamic allocation (grow as sequence generates)")

## Summary

You've mastered high-performance LLM systems:
- ✅ LLM inference challenges (memory, batching, latency)
- ✅ Continuous batching for higher throughput
- ✅ PagedAttention for efficient KV cache
- ✅ Throughput vs latency tradeoffs
- ✅ Production serving best practices

**Key Insights**:
1. **Memory is the bottleneck**: Not compute, but memory bandwidth
2. **Continuous batching**: Keep batch full by adding/removing requests dynamically
3. **PagedAttention**: Block-based KV cache eliminates fragmentation
4. **vLLM impact**: 2-4x throughput improvement over traditional serving
5. **Tradeoffs**: Balance throughput vs latency based on use case

**Performance Comparison**:

| System | Batching | KV Cache | Throughput | Latency |
|--------|----------|----------|------------|----------|
| Naive | Static | Contiguous | 1x | High |
| HuggingFace | Static | Contiguous | 1.5x | Medium |
| TGI | Continuous | Optimized | 2.5x | Medium |
| vLLM | Continuous | Paged | 4x | Low |

**Real-World Usage**:
- **vLLM**: Open-source, widely adopted, best throughput
- **TGI**: Hugging Face's high-performance inference
- **TensorRT-LLM**: NVIDIA's optimized engine
- **llama.cpp**: CPU inference, quantization focus

**Next Steps**:
- Quantization (INT8, INT4) for lower memory
- Speculative decoding for faster generation
- Distributed inference across multiple GPUs
- Model parallelism for very large models

**Next Notebook**: Model serving, quantization, and production deployment

## Exercises

1. **Implement continuous batching**: Build full continuous batching scheduler
2. **Compare static vs continuous**: Measure throughput difference
3. **PagedAttention simulation**: Implement block allocation/deallocation
4. **Memory analysis**: Calculate KV cache size for different models
5. **Optimize batch size**: Find optimal batch size for your hardware
6. **Build request queue**: Implement priority queue for request scheduling